# 🚀 YOLOv5-C2F 口罩检测训练
**使用自定义 C2f 模块 + Face Mask Detection 数据集**

> 逐个 Shift+Enter 运行，每个 cell 跑完再跑下一个

## 1. 检查 GPU

In [ ]:
!nvidia-smi

## 2. 克隆你的仓库 + 装环境

In [ ]:
# 挂载 Google Drive（保存训练结果）
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 克隆你的仓库
!git clone https://github.com/qianz7884-blip/yolo.git
%cd yolo

In [ ]:
# 修补 c2f 模块的 bug（变量 e 未定义）
!sed -i 's/int(c2\*e)/int(c2*0.5)/' models/common.py
print('✅ Bug 已修补')

# 安装依赖
!pip install -r requirements.txt -q
print('✅ 依赖安装完成')

## 3. 下载数据集 + 转换 YOLO 格式

In [ ]:
!pip install kagglehub -q

In [ ]:
import xml.etree.ElementTree as ET
import shutil, random
from pathlib import Path
import kagglehub

print('📥 下载数据集中...')
src = Path(kagglehub.dataset_download('andrewmvd/face-mask-detection'))

# 创建目录
dst = Path('/content/datasets/mask')
for s in ['train', 'val']:
    (dst / 'images' / s).mkdir(parents=True, exist_ok=True)
    (dst / 'labels' / s).mkdir(parents=True, exist_ok=True)

CLASS_MAP = {'with_mask': 0, 'without_mask': 1, 'mask_weared_incorrect': 2}

# VOC → YOLO
all_data = []
for xml_path in sorted((src / 'annotations').glob('*.xml')):
    tree = ET.parse(xml_path)
    root = tree.getroot()
    filename = root.find('filename').text
    size = root.find('size')
    w, h = int(size.find('width').text), int(size.find('height').text)
    objs = []
    for obj in root.findall('object'):
        name = obj.find('name').text.strip()
        if name not in CLASS_MAP: continue
        bbox = obj.find('bndbox')
        xc = ((float(bbox.find('xmin').text) + float(bbox.find('xmax').text)) / 2) / w
        yc = ((float(bbox.find('ymin').text) + float(bbox.find('ymax').text)) / 2) / h
        bw = (float(bbox.find('xmax').text) - float(bbox.find('xmin').text)) / w
        bh = (float(bbox.find('ymax').text) - float(bbox.find('ymin').text)) / h
        objs.append(f'{CLASS_MAP[name]} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}')
    all_data.append({'filename': filename, 'objects': objs})

random.seed(42)
random.shuffle(all_data)
n = int(len(all_data) * 0.8)

for split, data in [('train', all_data[:n]), ('val', all_data[n:])]:
    for item in data:
        stem = Path(item['filename']).stem
        img = src / 'images' / item['filename']
        if img.exists():
            shutil.copy2(img, dst / 'images' / split / item['filename'])
        lbl = dst / 'labels' / split / f'{stem}.txt'
        with open(lbl, 'w') as f:
            if item['objects']:
                f.write('\n'.join(item['objects']) + '\n')

print(f'✅ 训练集 {n} 张, 验证集 {len(all_data)-n} 张')

## 4. 训练（C2f 模型）

In [ ]:
!python train.py \
    --img 640 \
    --batch 16 \
    --epochs 100 \
    --data data/mask.yaml \
    --weights yolov5s.pt \
    --cfg models/yolov5s-c2f.yaml \
    --project /content/drive/MyDrive/yolov5-mask \
    --name exp-c2f \
    --workers 2

## 5. 查看训练结果

In [ ]:
from IPython.display import Image, display
exp = '/content/drive/MyDrive/yolov5-mask/exp-c2f'

print('📊 混淆矩阵:');   display(Image(filename=f'{exp}/confusion_matrix.png'))
print('📈 训练曲线:');   display(Image(filename=f'{exp}/results.png'))
print('🔍 验证样张 1:'); display(Image(filename=f'{exp}/val_batch0_pred.jpg'))
print('🔍 验证样张 2:'); display(Image(filename=f'{exp}/val_batch1_pred.jpg'))

## 6. 推理检测

In [ ]:
# 上传你自己的图片做口罩检测
from google.colab import files
import glob

uploaded = files.upload()
for fname in uploaded.keys():
    !python detect.py \
        --weights /content/drive/MyDrive/yolov5-mask/exp-c2f/weights/best.pt \
        --source {fname} \
        --conf 0.25
    result = glob.glob('/content/yolo/runs/detect/*/' + fname)
    if result: display(Image(filename=result[0]))

## 7. 下载模型

In [ ]:
from google.colab import files
files.download('/content/drive/MyDrive/yolov5-mask/exp-c2f/weights/best.pt')